# FIFA World Cup 2026 predictions

This submission uses completed international matches through March 31, 2026. It combines regularized Poisson score distributions, gradient-boosted home/away goal models, and a gradient-boosted win/draw/loss classifier. Match importance, rolling form, rolling attack/defense, Elo strength, and exponential recency weighting are calculated without future-match leakage. Separate trained models predict corners, yellow cards, red-card probability, and penalty-shootout probability. Official FIFA final squads contribute age, position, goalkeeper, club-league quality, and optional injury features. Validated market odds, international match-statistics, and referee override files are blended automatically when populated. USA, Canada, and Mexico receive a modest host-strength adjustment. Predicted integers maximize expected competition points rather than simply rounding model means, and knockout score distributions include the extra-time period required by the competition rules. Fifty thousand complete tournaments were simulated using FIFA's head-to-head group tiebreakers and exact 495-row Annex C mapping. A coherent bracket was selected to maximize expected matchup and winner points after applying the knockout-round multipliers.

The six playoff placeholders have been replaced by Bosnia and Herzegovina, Sweden, Turkey, Czech Republic, DR Congo, and Iraq.

In [ ]:
from io import StringIO
import pandas as pd


## Model selection

Selected regularization `alpha=0.003` and a `2.5`-year recency half-life. The historical mean competition score was `25.43` points per match for the Poisson tuning stage. Chronological validation across nine major-tournament windows selected goal blend `0.15` and outcome-classifier blend `0.00`, improving mean score/outcome points from `27.44` to `27.56` per match.

## Group stage predictions

In [ ]:
group_predictions = pd.read_csv(StringIO(r'''
match_id,group,home_team,away_team,date_utc,venue,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,winning_team
1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City",1,0,10,3,0,home
2,A,South Korea,Czech Republic,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara",1,1,9,3,0,home
3,B,Canada,Bosnia and Herzegovina,2026-06-12T19:00:00Z,"BMO Field, Toronto",2,1,10,4,0,home
4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles",1,1,9,5,0,away
5,D,Australia,Turkey,2026-06-13T04:00:00Z,"BC Place, Vancouver",1,1,9,2,0,away
6,B,Qatar,Switzerland,2026-06-13T19:00:00Z,"Levi's Stadium, Santa Clara",0,2,10,4,0,away
7,C,Brazil,Morocco,2026-06-13T22:00:00Z,"MetLife Stadium, East Rutherford",1,1,9,3,0,home
8,C,Haiti,Scotland,2026-06-14T01:00:00Z,"Gillette Stadium, Boston",1,2,10,4,0,away
9,E,Germany,Curaçao,2026-06-14T17:00:00Z,"NRG Stadium, Houston",2,0,11,1,0,home
10,F,Netherlands,Japan,2026-06-14T20:00:00Z,"AT&T Stadium, Dallas",1,1,9,2,0,home
11,E,Côte d'Ivoire,Ecuador,2026-06-14T23:00:00Z,"Lincoln Financial Field, Philadelphia",0,1,9,3,0,away
12,F,Sweden,Tunisia,2026-06-15T02:00:00Z,"Estadio BBVA, Monterrey",1,1,9,3,0,home
13,H,Spain,Cabo Verde,2026-06-15T16:00:00Z,"Mercedes-Benz Stadium, Atlanta",2,0,11,3,0,home
14,G,Belgium,Egypt,2026-06-15T19:00:00Z,"Lumen Field, Seattle",1,1,9,4,0,home
15,H,Saudi Arabia,Uruguay,2026-06-15T22:00:00Z,"Hard Rock Stadium, Miami",0,1,9,3,0,away
16,G,Iran,New Zealand,2026-06-16T01:00:00Z,"SoFi Stadium, Los Angeles",2,1,10,5,0,home
17,J,Austria,Jordan,2026-06-16T04:00:00Z,"Levi's Stadium, Santa Clara",1,1,9,4,0,home
18,I,France,Senegal,2026-06-16T19:00:00Z,"MetLife Stadium, East Rutherford",1,1,9,4,0,home
19,I,Iraq,Norway,2026-06-16T22:00:00Z,"Gillette Stadium, Boston",0,1,10,4,0,away
20,J,Argentina,Algeria,2026-06-17T01:00:00Z,"GEHA Field at Arrowhead Stadium, Kansas City",2,0,10,3,0,home
21,K,Portugal,DR Congo,2026-06-17T17:00:00Z,"NRG Stadium, Houston",2,0,10,4,0,home
22,L,England,Croatia,2026-06-17T20:00:00Z,"AT&T Stadium, Dallas",1,1,9,3,0,home
23,L,Ghana,Panama,2026-06-17T23:00:00Z,"BMO Field, Toronto",1,1,9,3,0,away
24,K,Uzbekistan,Colombia,2026-06-18T02:00:00Z,"Estadio Azteca, Mexico City",0,1,9,4,0,away
25,A,Czech Republic,South Africa,2026-06-18T16:00:00Z,"Mercedes-Benz Stadium, Atlanta",1,1,9,5,0,home
26,B,Switzerland,Bosnia and Herzegovina,2026-06-18T19:00:00Z,"SoFi Stadium, Los Angeles",2,0,11,4,0,home
27,B,Canada,Qatar,2026-06-18T22:00:00Z,"BC Place, Vancouver",2,0,11,3,0,home
28,A,Mexico,South Korea,2026-06-19T01:00:00Z,"Estadio Akron, Guadalajara",1,1,9,3,0,home
29,D,Turkey,Paraguay,2026-06-19T04:00:00Z,"Levi's Stadium, Santa Clara",1,1,9,3,0,home
30,D,USA,Australia,2026-06-19T19:00:00Z,"Lumen Field, Seattle",1,1,9,3,0,away
31,C,Scotland,Morocco,2026-06-19T22:00:00Z,"Gillette Stadium, Boston",1,1,9,3,0,away
32,C,Brazil,Haiti,2026-06-20T01:00:00Z,"Lincoln Financial Field, Philadelphia",2,0,11,3,0,home
33,F,Tunisia,Japan,2026-06-20T04:00:00Z,"Estadio BBVA, Monterrey",0,1,9,4,0,away
34,F,Netherlands,Sweden,2026-06-20T17:00:00Z,"NRG Stadium, Houston",2,1,10,3,0,home
35,E,Germany,Côte d'Ivoire,2026-06-20T20:00:00Z,"BMO Field, Toronto",2,1,10,4,0,home
36,E,Ecuador,Curaçao,2026-06-21T00:00:00Z,"GEHA Field at Arrowhead Stadium, Kansas City",2,0,10,4,0,home
37,H,Spain,Saudi Arabia,2026-06-21T16:00:00Z,"Mercedes-Benz Stadium, Atlanta",2,0,11,3,0,home
38,G,Belgium,Iran,2026-06-21T19:00:00Z,"SoFi Stadium, Los Angeles",1,1,10,3,0,home
39,H,Uruguay,Cabo Verde,2026-06-21T22:00:00Z,"Hard Rock Stadium, Miami",1,0,10,3,0,home
40,G,New Zealand,Egypt,2026-06-22T01:00:00Z,"BC Place, Vancouver",1,1,9,3,0,away
41,J,Argentina,Austria,2026-06-22T17:00:00Z,"AT&T Stadium, Dallas",1,0,10,5,0,home
42,I,France,Iraq,2026-06-22T21:00:00Z,"Lincoln Financial Field, Philadelphia",2,0,10,3,0,home
43,I,Norway,Senegal,2026-06-23T00:00:00Z,"MetLife Stadium, East Rutherford",1,1,9,2,0,home
44,J,Jordan,Algeria,2026-06-23T03:00:00Z,"Levi's Stadium, Santa Clara",1,1,9,2,0,away
45,K,Portugal,Uzbekistan,2026-06-23T17:00:00Z,"NRG Stadium, Houston",2,1,9,5,0,home
46,L,England,Ghana,2026-06-23T20:00:00Z,"Gillette Stadium, Boston",2,0,11,3,0,home
47,L,Panama,Croatia,2026-06-23T23:00:00Z,"BMO Field, Toronto",1,1,9,3,0,away
48,K,Colombia,DR Congo,2026-06-24T02:00:00Z,"Estadio Akron, Guadalajara",1,0,10,4,0,home
49,B,Switzerland,Canada,2026-06-24T19:00:00Z,"BC Place, Vancouver",1,1,10,3,0,home
50,B,Bosnia and Herzegovina,Qatar,2026-06-24T19:00:00Z,"Lumen Field, Seattle",1,1,10,4,0,home
51,C,Scotland,Brazil,2026-06-24T22:00:00Z,"Hard Rock Stadium, Miami",1,1,9,4,0,away
52,C,Morocco,Haiti,2026-06-24T22:00:00Z,"Mercedes-Benz Stadium, Atlanta",2,0,11,4,0,home
53,A,Czech Republic,Mexico,2026-06-25T01:00:00Z,"Estadio Azteca, Mexico City",1,1,9,4,0,away
54,A,South Africa,South Korea,2026-06-25T01:00:00Z,"Estadio BBVA, Monterrey",1,1,9,4,0,away
55,E,Curaçao,Côte d'Ivoire,2026-06-25T20:00:00Z,"Lincoln Financial Field, Philadelphia",0,1,9,4,0,away
56,E,Ecuador,Germany,2026-06-25T20:00:00Z,"MetLife Stadium, East Rutherford",1,1,9,2,0,away
57,F,Japan,Sweden,2026-06-25T23:00:00Z,"AT&T Stadium, Dallas",2,1,10,3,0,home
58,F,Tunisia,Netherlands,2026-06-25T23:00:00Z,"GEHA Field at Arrowhead Stadium, Kansas City",0,1,9,3,0,away
59,D,USA,Turkey,2026-06-26T02:00:00Z,"SoFi Stadium, Los Angeles",1,1,9,3,0,away
60,D,Paraguay,Australia,2026-06-26T02:00:00Z,"Levi's Stadium, Santa Clara",1,1,9,2,0,away
61,I,Norway,France,2026-06-26T19:00:00Z,"Gillette Stadium, Boston",1,1,9,3,0,away
62,I,Senegal,Iraq,2026-06-26T19:00:00Z,"BMO Field, Toronto",1,0,10,4,0,home
63,H,Cabo Verde,Saudi Arabia,2026-06-27T00:00:00Z,"NRG Stadium, Houston",1,1,9,3,0,away
64,H,Uruguay,Spain,2026-06-27T00:00:00Z,"Estadio Akron, Guadalajara",0,1,9,4,0,away
65,G,Egypt,Iran,2026-06-27T03:00:00Z,"Lumen Field, Seattle",1,1,9,4,0,away
66,G,New Zealand,Belgium,2026-06-27T03:00:00Z,"BC Place, Vancouver",1,2,9,4,0,away
67,L,Panama,England,2026-06-27T21:00:00Z,"MetLife Stadium, East Rutherford",0,2,10,3,0,away
68,L,Croatia,Ghana,2026-06-27T21:00:00Z,"Lincoln Financial Field, Philadelphia",2,0,10,3,0,home
69,K,Colombia,Portugal,2026-06-27T23:30:00Z,"Hard Rock Stadium, Miami",1,1,9,3,0,away
70,K,DR Congo,Uzbekistan,2026-06-27T23:30:00Z,"Mercedes-Benz Stadium, Atlanta",1,1,9,3,0,away
71,J,Algeria,Austria,2026-06-28T02:00:00Z,"GEHA Field at Arrowhead Stadium, Kansas City",1,1,9,2,0,away
72,J,Jordan,Argentina,2026-06-28T02:00:00Z,"AT&T Stadium, Dallas",0,2,10,3,0,away
'''))
group_predictions


## Knockout stage predictions

In [ ]:
knockout_predictions = pd.read_csv(StringIO(r'''
match_id,round,multiplier,date_utc,venue,slot_home,slot_away,predicted_home_team,predicted_away_team,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,match_winner,penalties
73,Round of 32,1,2026-06-28T19:00:00Z,"SoFi Stadium, Los Angeles",Runner-up Group A,Runner-up Group B,Mexico,Canada,1,0,9,3,0,home,False
74,Round of 32,1,2026-06-29T17:00:00Z,"NRG Stadium, Houston",Winner Group E,Best 3rd (Groups A/B/C/D/F),Germany,Japan,1,2,10,1,0,away,False
75,Round of 32,1,2026-06-29T20:30:00Z,"Gillette Stadium, Boston",Winner Group F,Runner-up Group C,Netherlands,Morocco,2,1,9,3,0,home,False
76,Round of 32,1,2026-06-30T01:00:00Z,"Estadio BBVA, Monterrey",Winner Group C,Runner-up Group F,Brazil,Tunisia,1,0,10,3,0,home,False
77,Round of 32,1,2026-06-30T17:00:00Z,"AT&T Stadium, Dallas",Winner Group I,Best 3rd (Groups C/D/F/G/H),France,Egypt,1,0,10,3,0,home,False
78,Round of 32,1,2026-06-30T21:00:00Z,"MetLife Stadium, East Rutherford",Runner-up Group E,Runner-up Group I,Ecuador,Senegal,1,0,9,4,0,home,False
79,Round of 32,1,2026-07-01T01:00:00Z,"Estadio Azteca, Mexico City",Winner Group A,Best 3rd (Groups C/E/F/H/I),South Korea,Norway,2,1,9,3,0,home,False
80,Round of 32,1,2026-07-01T16:00:00Z,"Mercedes-Benz Stadium, Atlanta",Winner Group L,Best 3rd (Groups E/H/I/J/K),England,DR Congo,1,0,10,3,0,home,False
81,Round of 32,1,2026-07-01T20:00:00Z,"Lumen Field, Seattle",Winner Group D,Best 3rd (Groups B/E/F/I/J),Paraguay,Bosnia and Herzegovina,1,0,9,4,0,home,False
82,Round of 32,1,2026-07-02T00:00:00Z,"Levi's Stadium, Santa Clara",Winner Group G,Best 3rd (Groups A/E/H/I/J),Belgium,South Africa,2,1,11,3,0,home,False
83,Round of 32,1,2026-07-02T19:00:00Z,"BMO Field, Toronto",Runner-up Group K,Runner-up Group L,Colombia,Croatia,1,2,9,3,0,away,False
84,Round of 32,1,2026-07-02T23:00:00Z,"SoFi Stadium, Los Angeles",Winner Group H,Runner-up Group J,Spain,Algeria,2,1,11,3,0,home,False
85,Round of 32,1,2026-07-03T03:00:00Z,"BC Place, Vancouver",Winner Group B,Best 3rd (Groups E/F/G/I/J),Switzerland,Jordan,2,1,10,5,0,home,False
86,Round of 32,1,2026-07-03T18:00:00Z,"AT&T Stadium, Dallas",Winner Group J,Runner-up Group H,Argentina,Uruguay,1,0,9,3,0,home,False
87,Round of 32,1,2026-07-03T22:00:00Z,"Hard Rock Stadium, Miami",Winner Group K,Best 3rd (Groups D/E/I/J/L),Portugal,Panama,2,1,10,3,0,home,False
88,Round of 32,1,2026-07-04T01:30:00Z,"GEHA Field at Arrowhead Stadium, Kansas City",Runner-up Group D,Runner-up Group G,Turkey,Iran,1,2,10,3,0,away,False
89,Round of 16,2,2026-07-04T17:00:00Z,"NRG Stadium, Houston",Winner Match 74,Winner Match 77,Japan,France,1,2,9,3,0,away,False
90,Round of 16,2,2026-07-04T21:00:00Z,"Lincoln Financial Field, Philadelphia",Winner Match 73,Winner Match 75,Mexico,Netherlands,1,2,9,3,0,away,False
91,Round of 16,2,2026-07-05T20:00:00Z,"MetLife Stadium, East Rutherford",Winner Match 76,Winner Match 78,Brazil,Ecuador,1,0,9,4,0,home,False
92,Round of 16,2,2026-07-06T00:00:00Z,"Estadio Azteca, Mexico City",Winner Match 79,Winner Match 80,South Korea,England,1,2,9,4,0,away,False
93,Round of 16,2,2026-07-06T19:00:00Z,"AT&T Stadium, Dallas",Winner Match 83,Winner Match 84,Croatia,Spain,1,2,10,3,0,away,False
94,Round of 16,2,2026-07-07T00:00:00Z,"Lumen Field, Seattle",Winner Match 81,Winner Match 82,Paraguay,Belgium,1,2,9,3,0,away,False
95,Round of 16,2,2026-07-07T16:00:00Z,"Mercedes-Benz Stadium, Atlanta",Winner Match 86,Winner Match 88,Argentina,Iran,1,0,10,4,0,home,False
96,Round of 16,2,2026-07-07T20:00:00Z,"BC Place, Vancouver",Winner Match 85,Winner Match 87,Switzerland,Portugal,1,2,9,2,0,away,False
97,Quarter-final,4,2026-07-09T20:00:00Z,"Gillette Stadium, Boston",Winner Match 89,Winner Match 90,France,Netherlands,1,2,10,3,0,away,False
98,Quarter-final,4,2026-07-10T19:00:00Z,"SoFi Stadium, Los Angeles",Winner Match 93,Winner Match 94,Spain,Belgium,2,1,10,3,0,home,False
99,Quarter-final,4,2026-07-11T21:00:00Z,"Hard Rock Stadium, Miami",Winner Match 91,Winner Match 92,Brazil,England,1,2,9,4,0,away,False
100,Quarter-final,4,2026-07-12T01:00:00Z,"GEHA Field at Arrowhead Stadium, Kansas City",Winner Match 95,Winner Match 96,Argentina,Portugal,2,1,9,3,0,home,False
101,Semi-final,8,2026-07-14T19:00:00Z,"AT&T Stadium, Dallas",Winner Match 97,Winner Match 98,Netherlands,Spain,1,2,10,4,0,away,False
102,Semi-final,8,2026-07-15T19:00:00Z,"Mercedes-Benz Stadium, Atlanta",Winner Match 99,Winner Match 100,England,Argentina,0,1,9,4,0,away,False
103,Third-place playoff,8,2026-07-18T21:00:00Z,"Hard Rock Stadium, Miami",Loser Match 101,Loser Match 102,Netherlands,England,2,1,9,2,0,home,False
104,Final,16,2026-07-19T19:00:00Z,"MetLife Stadium, East Rutherford",Winner Match 101,Winner Match 102,Spain,Argentina,2,1,9,2,0,home,False
'''))
knockout_predictions


## Submission validation

In [ ]:
assert len(group_predictions) == 72
assert len(knockout_predictions) == 32
assert not group_predictions.isna().any().any()
assert not knockout_predictions.isna().any().any()
assert set(group_predictions['winning_team']) <= {'home', 'away', 'draw'}
assert set(knockout_predictions['match_winner']) <= {'home', 'away'}
assert set(knockout_predictions['penalties']) <= {True, False}
knockout_score_winner = knockout_predictions.apply(
    lambda row: 'home' if row.predicted_home_goals > row.predicted_away_goals else 'away', axis=1
)
assert (knockout_score_winner == knockout_predictions['match_winner']).all()
print('Validated all 104 predictions with no missing values.')
